In [1]:
import sys
sys.path.insert(0, '../src/data')

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')

In [ ]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment('phase3_transfer')

In [2]:
office_files = list(PROCESSED_DIR.glob('office001_*.parquet'))
print(f"office001 stations found: {len(office_files)}")
for f in sorted(office_files):
    print(f"  {f.stem}")

office001 stations found: 8
  office001_19-102-260-1633
  office001_19-102-260-1634
  office001_19-102-260-1635
  office001_19-102-260-1636
  office001_19-102-260-1637
  office001_19-102-260-1638
  office001_19-102-260-1639
  office001_19-102-260-1640


In [3]:
records = []
for f in sorted(office_files):
    df = pd.read_parquet(f)
    df = df.sort_values('timestamp')
    total_hours = len(df)
    total_weeks = total_hours / (24 * 7)
    nonzero_hours = (df['sessions'] > 0).sum()
    records.append({
        'station': f.stem,
        'start': df['timestamp'].min(),
        'end':   df['timestamp'].max(),
        'total_weeks': round(total_weeks, 1),
        'nonzero_hours': nonzero_hours,
        'sparsity_pct': round(100 * nonzero_hours / total_hours, 1)
    })

coverage = pd.DataFrame(records)
print(coverage.to_string(index=False))

                  station                     start                       end  total_weeks  nonzero_hours  sparsity_pct
office001_19-102-260-1633 2019-03-25 09:00:00-07:00 2020-10-12 10:00:00-07:00         81.0           1940          14.3
office001_19-102-260-1634 2019-03-25 10:00:00-07:00 2020-10-30 17:00:00-07:00         83.6           1003           7.1
office001_19-102-260-1635 2019-03-27 17:00:00-07:00 2020-11-20 17:00:00-08:00         86.3            909           6.3
office001_19-102-260-1636 2019-03-29 11:00:00-07:00 2020-09-10 19:00:00-07:00         75.9           1221           9.6
office001_19-102-260-1637 2019-05-24 13:00:00-07:00 2020-07-24 17:00:00-07:00         61.0            372           3.6
office001_19-102-260-1638 2019-04-01 10:00:00-07:00 2020-03-06 18:00:00-08:00         48.6            598           7.3
office001_19-102-260-1639 2019-04-12 13:00:00-07:00 2020-12-29 17:00:00-08:00         89.6           1667          11.1
office001_19-102-260-1640 2019-03-25 14:

In [4]:
caltech_files = list(PROCESSED_DIR.glob('caltech_*.parquet'))
jpl_files = list(PROCESSED_DIR.glob('jpl_*.parquet'))

print(f"Caltech stations: {len(caltech_files)}")
print(f"JPL stations:     {len(jpl_files)}")

# Check columns on one sample file
sample = pd.read_parquet(caltech_files[0])
print(f"\nColumns: {list(sample.columns)}")
print(f"Shape:   {sample.shape}")
print(f"\nSample rows:")
print(sample.head(3).to_string())

Caltech stations: 55
JPL stations:     52

Columns: ['timestamp', 'station_id', 'site', 'sessions', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'lag_1h', 'lag_24h', 'lag_168h', 'rolling_24h_mean', 'rolling_7d_mean']
Shape:   (18217, 14)

Sample rows:
                  timestamp           station_id     site  sessions  hour  day_of_week  month  is_weekend  is_holiday  lag_1h  lag_24h  lag_168h  rolling_24h_mean  rolling_7d_mean
0 2018-05-01 10:00:00-07:00  caltech_2-39-123-23  caltech         1    10            1      5           0           0     NaN      NaN       NaN               NaN              NaN
1 2018-05-01 11:00:00-07:00  caltech_2-39-123-23  caltech         1    11            1      5           0           0     1.0      NaN       NaN               1.0              1.0
2 2018-05-01 12:00:00-07:00  caltech_2-39-123-23  caltech         1    12            1      5           0           0     1.0      NaN       NaN               1.0              1.0


In [1]:
import sys
sys.path.insert(0, '../src/models')
from global_model import load_training_stations, train_global_model, save_model
print("Loading training data...")
df_train = load_training_stations(sites=('caltech', 'jpl'))
print("\nTraining global model...")
booster, params = train_global_model(df_train, num_boost_round=300)
print("\nSaving...")
save_model(booster)
print("\nFeature importances:")
import pandas as pd
fi = pd.Series(
    booster.feature_importance(importance_type='gain'),
    index=booster.feature_name()
).sort_values(ascending=False)
print(fi)

Loading training data...
  caltech: 55 stations
  jpl: 52 stations
  Total rows: 2,020,364  |  Stations: 107

Training global model...
  Training on 2,020,364 rows, 11 features

Saving...
  Model saved to C:\Users\rksan_amz5yv3\ev-coldstart-forecast\models\global_model.pkl

Feature importances:
lag_1h              1.856706e+07
hour                6.350385e+06
lag_168h            4.663992e+06
rolling_24h_mean    3.886718e+06
day_of_week         3.319562e+06
lag_24h             2.935085e+06
rolling_7d_mean     1.742440e+06
site_encoded        1.298946e+06
month               3.266394e+05
is_weekend          2.004883e+05
is_holiday          1.770664e+05
dtype: float64


In [1]:
import sys
sys.path.insert(0, '../src/models')

from global_model import load_model
from transfer import load_station, temporal_split, fine_tune, predict

# Load global model
booster = load_model()

# Pick one office001 station for the smoke test
station_id = 'office001_19-102-260-1633'
site       = 'office001'

# Load and split at 2 weeks
df = load_station(station_id, site)
train_df, test_df = temporal_split(df, train_weeks=2)

print(f"Station:    {station_id}")
print(f"Train rows: {len(train_df)}  ({len(train_df)/168:.1f} weeks)")
print(f"Test rows:  {len(test_df)}  ({len(test_df)/168:.1f} weeks)")

# Fine-tune
ft_booster = fine_tune(booster, train_df)

# Predict on test set
preds = predict(ft_booster, test_df)
actuals = test_df['sessions'].values

# Quick MAE check
mae = abs(preds - actuals).mean()
print(f"\nFine-tuned MAE on test set: {mae:.4f}")
print(f"Actual mean sessions:        {actuals.mean():.4f}")
print(f"Predicted mean sessions:     {preds.mean():.4f}")

Station:    office001_19-102-260-1633
Train rows: 336  (2.0 weeks)
Test rows:  13274  (79.0 weeks)

Fine-tuned MAE on test set: 0.0351
Actual mean sessions:        0.1444
Predicted mean sessions:     0.1361


In [9]:
import sys
sys.path.insert(0, '../src/models')
sys.path.insert(0, '../src/evaluation')

import mlflow
import mlflow.lightgbm
import pandas as pd
import numpy as np
from pathlib import Path

from global_model import load_model
from transfer import load_station, temporal_split, fine_tune, predict
from metrics import mae, rmse, mape

PROCESSED_DIR = Path('../data/processed')
SITE_MAP = {'caltech': 0, 'jpl': 1, 'office001': 2}

office_stations = [f.stem for f in sorted(PROCESSED_DIR.glob('office001_*.parquet'))]
data_volumes    = [1, 2, 3]   # weeks of fine-tuning data

print(f"Stations : {len(office_stations)}")
print(f"Volumes  : {data_volumes}")
print(f"Total runs: {len(office_stations) * len(data_volumes)}")

Stations : 8
Volumes  : [1, 2, 3]
Total runs: 24


In [10]:
booster = load_model()

mlflow.set_experiment('phase3_transfer')

results = []

for weeks in data_volumes:
    week_maes = []

    for station_id in office_stations:
        with mlflow.start_run(run_name=f"transfer_{station_id}_w{weeks}"):

            # --- load and split ---
            df = load_station(station_id, 'office001')
            train_df, test_df = temporal_split(df, train_weeks=weeks)

            # guard: skip if test set is too small to evaluate meaningfully
            if len(test_df) < 168:
                print(f"  SKIP {station_id} @ {weeks}w — test set only {len(test_df)} rows")
                mlflow.end_run(status='FAILED')
                continue

            # --- fine-tune ---
            ft_booster = fine_tune(booster, train_df)

            # --- evaluate ---
            preds   = predict(ft_booster, test_df)
            actuals = test_df['sessions'].values

            station_mae  = mae(actuals, preds)
            station_rmse = rmse(actuals, preds)
            station_mape = mape(actuals, preds)

            week_maes.append(station_mae)

            # --- log to MLflow ---
            mlflow.log_params({
                'station_id':       station_id,
                'train_weeks':      weeks,
                'train_rows':       len(train_df),
                'test_rows':        len(test_df),
                'num_boost_round':  100,
                'learning_rate':    0.01,
                'model_type':       'transfer'
            })
            mlflow.log_metrics({
                'mae':  station_mae,
                'rmse': station_rmse,
                'mape': station_mape
            })

            print(f"  {station_id} | {weeks}w | MAE {station_mae:.4f} | RMSE {station_rmse:.4f}")

    avg_mae = np.mean(week_maes) if week_maes else float('nan')
    results.append({'train_weeks': weeks, 'avg_mae': avg_mae, 'n_stations': len(week_maes)})
    print(f"\n>>> {weeks} week(s) — avg MAE across {len(week_maes)} stations: {avg_mae:.4f}\n")

  office001_19-102-260-1633 | 1w | MAE 0.0349 | RMSE 0.1618
  office001_19-102-260-1634 | 1w | MAE 0.0244 | RMSE 0.1417
  office001_19-102-260-1635 | 1w | MAE 0.0187 | RMSE 0.1223
  office001_19-102-260-1636 | 1w | MAE 0.0309 | RMSE 0.1476
  office001_19-102-260-1637 | 1w | MAE 0.0141 | RMSE 0.1029
  office001_19-102-260-1638 | 1w | MAE 0.0264 | RMSE 0.1473
  office001_19-102-260-1639 | 1w | MAE 0.0295 | RMSE 0.1503
  office001_19-102-260-1640 | 1w | MAE 0.0441 | RMSE 0.1706

>>> 1 week(s) — avg MAE across 8 stations: 0.0279

  office001_19-102-260-1633 | 2w | MAE 0.0351 | RMSE 0.1617
  office001_19-102-260-1634 | 2w | MAE 0.0250 | RMSE 0.1410
  office001_19-102-260-1635 | 2w | MAE 0.0189 | RMSE 0.1232
  office001_19-102-260-1636 | 2w | MAE 0.0366 | RMSE 0.1454
  office001_19-102-260-1637 | 2w | MAE 0.0143 | RMSE 0.1037
  office001_19-102-260-1638 | 2w | MAE 0.0267 | RMSE 0.1480
  office001_19-102-260-1639 | 2w | MAE 0.0298 | RMSE 0.1509
  office001_19-102-260-1640 | 2w | MAE 0.0453 | 

In [11]:
# Final comparison table
results_df = pd.DataFrame(results)

# Phase 2 baselines for reference (paste your actual numbers here)
baseline_lgbm  = {1: 0.1024, 2: 0.089, 3: 0.0864}
baseline_naive   = {1: 0.3488, 2: 0.3438, 3: 0.3481}

results_df['seasonal_naive_mae'] = results_df['train_weeks'].map(baseline_naive)
results_df['vanilla_lgbm_mae']   = results_df['train_weeks'].map(baseline_lgbm)

print("\n=== Phase 3 Ablation Results ===")
print(results_df.to_string(index=False))


=== Phase 3 Ablation Results ===
 train_weeks  avg_mae  n_stations  seasonal_naive_mae  vanilla_lgbm_mae
           1 0.027878           8              0.3488            0.1024
           2 0.028970           8              0.3438            0.0890
           3 0.028417           8              0.3481            0.0864


In [7]:
import inspect
print(inspect.getsource(LightGBMBaseline.fit))

    def fit(self, train_df):
        clean = train_df[FEATURE_COLS + [TARGET_COL]].copy()
        clean[FEATURE_COLS] = clean[FEATURE_COLS].fillna(0)
        clean = clean.dropna(subset=[TARGET_COL])
        X = clean[FEATURE_COLS].values
        y = clean[TARGET_COL].values
        train_data = lgb.Dataset(X, label=y, feature_name=FEATURE_COLS)
        self.model = lgb.train(self.params, train_data, num_boost_round=self.n_estimators)
        return self



In [8]:
print(inspect.getsource(LightGBMBaseline.predict))

    def predict(self, eval_df):
        X = eval_df[FEATURE_COLS].fillna(0).values
        return self.model.predict(X)



In [12]:
from baseline import LightGBMBaseline

vanilla_results = []

for weeks in data_volumes:
    week_maes = []

    for station_id in office_stations:
        with mlflow.start_run(run_name=f"vanilla_lgbm_{station_id}_w{weeks}"):

            df = load_station(station_id, 'office001')
            train_df, test_df = temporal_split(df, train_weeks=weeks)

            if len(test_df) < 168:
                mlflow.end_run(status='FAILED')
                continue

            model = LightGBMBaseline()
            model.fit(train_df)
            preds   = np.clip(model.predict(test_df), 0, None)
            actuals = test_df['sessions'].values

            station_mae  = mae(actuals, preds)
            station_rmse = rmse(actuals, preds)
            station_mape = mape(actuals, preds)

            week_maes.append(station_mae)

            mlflow.log_params({
                'station_id':  station_id,
                'train_weeks': weeks,
                'train_rows':  len(train_df),
                'test_rows':   len(test_df),
                'model_type':  'vanilla_lgbm'
            })
            mlflow.log_metrics({
                'mae':  station_mae,
                'rmse': station_rmse,
                'mape': station_mape
            })

            print(f"  {station_id} | {weeks}w | MAE {station_mae:.4f}")

    avg_mae = np.mean(week_maes) if week_maes else float('nan')
    vanilla_results.append({'train_weeks': weeks, 'vanilla_office001_mae': avg_mae})
    print(f"\n>>> {weeks} week(s) — vanilla avg MAE: {avg_mae:.4f}\n")

  office001_19-102-260-1633 | 1w | MAE 0.0334
  office001_19-102-260-1634 | 1w | MAE 0.0231
  office001_19-102-260-1635 | 1w | MAE 0.0634
  office001_19-102-260-1636 | 1w | MAE 0.0264
  office001_19-102-260-1637 | 1w | MAE 0.0110
  office001_19-102-260-1638 | 1w | MAE 0.0239
  office001_19-102-260-1639 | 1w | MAE 0.0291
  office001_19-102-260-1640 | 1w | MAE 0.0299

>>> 1 week(s) — vanilla avg MAE: 0.0300

  office001_19-102-260-1633 | 2w | MAE 0.0330
  office001_19-102-260-1634 | 2w | MAE 0.0211
  office001_19-102-260-1635 | 2w | MAE 0.0709
  office001_19-102-260-1636 | 2w | MAE 0.0539
  office001_19-102-260-1637 | 2w | MAE 0.0112
  office001_19-102-260-1638 | 2w | MAE 0.0725
  office001_19-102-260-1639 | 2w | MAE 0.0294
  office001_19-102-260-1640 | 2w | MAE 0.0299

>>> 2 week(s) — vanilla avg MAE: 0.0402

  office001_19-102-260-1633 | 3w | MAE 0.0414
  office001_19-102-260-1634 | 3w | MAE 0.0213
  office001_19-102-260-1635 | 3w | MAE 0.0787
  office001_19-102-260-1636 | 3w | MAE 0.0

In [13]:
vanilla_df = pd.DataFrame(vanilla_results)

final_df = results_df[['train_weeks', 'avg_mae']].merge(vanilla_df, on='train_weeks')
final_df.columns = ['train_weeks', 'transfer_mae', 'vanilla_lgbm_office001_mae']
final_df['seasonal_naive_mae'] = final_df['train_weeks'].map(baseline_naive)

print("\n=== Phase 3 Final Ablation Table (all evaluated on office001) ===")
print(final_df.to_string(index=False))
print("\nSpeedup (vanilla / transfer):")
print((final_df['vanilla_lgbm_office001_mae'] / final_df['transfer_mae']).round(2).values)


=== Phase 3 Final Ablation Table (all evaluated on office001) ===
 train_weeks  transfer_mae  vanilla_lgbm_office001_mae  seasonal_naive_mae
           1      0.027878                    0.030022              0.3488
           2      0.028970                    0.040240              0.3438
           3      0.028417                    0.035343              0.3481

Speedup (vanilla / transfer):
[1.08 1.39 1.24]
